In [ ]:
# PREPARACIÓN DEL ENTORNO


import os
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import glob
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# Configuración de dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"

# Local data are not distributed with this repository.
# Run Jupyter from the repository root; DENTAL_DATA_DIR can override the path.
BASE_DIR = str(Path(os.environ.get("DENTAL_DATA_DIR", "data/dataset_final")).resolve())
for split in ("train", "val", "test"):
    for kind in ("images", "labels", "masks"):
        if not (Path(BASE_DIR) / split / kind).is_dir():
            raise FileNotFoundError(f"Missing {split}/{kind} in {BASE_DIR}; see data/README.md")
    images = sorted((Path(BASE_DIR) / split / "images").glob("*.jpg"))
    if not images:
        raise ValueError(f"Empty image split: {split}")
    for image_path in images:
        for kind, suffix in (("labels", ".txt"), ("masks", ".png")):
            pair = Path(BASE_DIR) / split / kind / (image_path.stem + suffix)
            if not pair.is_file():
                raise FileNotFoundError(f"Missing paired annotation: {pair}")

# Seeds apply to new experiments, not the archived run.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# FUNCIONES AUXILIARES
def plot_training_curves(history, title="Curvas de Entrenamiento"):
    """Grafica las pérdidas de entrenamiento y validación"""
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title(title)
    plt.xlabel('Épocas')
    plt.ylabel('Loss (Pérdida)')
    plt.legend()
    plt.grid(True)
    plt.show()

def show_segmentation_result(model, dataset, index=0):
    """Visualiza: Imagen Original | Máscara Real | Máscara Predicha"""
    model.eval()
    image, mask = dataset[index]
    input_tensor = image.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(input_tensor)
        pred = torch.sigmoid(pred) # Probabilidad 0-1
        pred = (pred > 0.5).float() # Binarizar

    # Des-normalizar para visualizar
    inv_normalize = transforms.Normalize(
       mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
       std=[1/0.229, 1/0.224, 1/0.225]
    )
    img_show = inv_normalize(image).permute(1, 2, 0).cpu().numpy()
    img_show = np.clip(img_show, 0, 1)

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1); plt.imshow(img_show); plt.title("Imagen (Test)")
    plt.subplot(1, 3, 2); plt.imshow(mask[0], cmap='gray'); plt.title("MATLAB pseudo-label")
    plt.subplot(1, 3, 3); plt.imshow(pred[0,0].cpu(), cmap='gray'); plt.title("Predicción U-Net")
    plt.show()

# Función para calcular Accuracy 
def calculate_accuracy(preds, masks):
    preds_bin = (torch.sigmoid(preds) > 0.5).float()
    correct = (preds_bin == masks).sum().item()
    total = torch.numel(masks)
    return correct / total

# Dental implant detection with YOLOv8n

Academic team project by Joel Alfaro and Daniel Pastor. Install requirements from the repository root before running this notebook. Provide local data as described in `data/README.md`. This notebook trains fresh models; it contains no original weights or image outputs. Archived results are in `results/` and are not reproduced by merely opening this notebook.

The detection target is one implant class. Bounding boxes and segmentation masks were generated automatically by MATLAB, so evaluation measures agreement with pseudo-labels. This is an educational computer-vision experiment, with no clinical validation.


In [ ]:
# TAREA A: DETECCIÓN DE OBJETOS (YOLOv8)
print("\n" + "="*40)
print(" INICIANDO DETECCIÓN (YOLOv8)")
print("="*40)

# Configuración YAML para YOLO
yaml_content = f"""
path: {BASE_DIR}
train: train/images
val: val/images
test: test/images

nc: 1
names: ['implante']
"""

with open("data_implantes.yaml", "w") as f:
    f.write(yaml_content)

# Carga y Entrenamiento
# YOLOv8n starts from COCO-pretrained weights; task suitability requires evaluation.
model_yolo = YOLO('yolov8n.pt')

print("... Entrenando YOLOv8...")
results_yolo = model_yolo.train(
    data='data_implantes.yaml',
    epochs=30,           # Ajustable
    imgsz=640,           # Tamaño estándar YOLO
    batch=8,
    seed=42,
    project='runs/detect',
    name='train_implante',
    exist_ok=True,
    verbose=False,

    # --- HIPERPARÁMETROS DE DATA AUGMENTATION ---
    fliplr=0.5,      # 50% probabilidad de volteo horizontal (Muy útil)
    flipud=0.0,      # 0% volteo vertical (No tiene sentido en dientes)
    degrees=10.0,    # Rotación leve (+/- 10 grados)
    scale=0.5,       # Zoom/Escala (+/- 50%)

    # Ajustes de "Color" (En X-ray es Brillo/Contraste)
    hsv_h=0.0,       # Hue (Matiz): 0.0 -> No cambiar el color (es escala de grises)
    hsv_s=0.0,       # Saturation: 0.0 -> No cambiar saturación
    hsv_v=0.4,       # Value (Brillo): Variar brillo un 40% (Vital para X-rays)

    # Técnicas avanzadas de YOLO
    mosaic=1.0,      # Activar Mosaic al 100% (Ayuda mucho en detección)
    mixup=0.1,       # Mezclar imágenes levemente (opcional, ayuda si tienes pocos datos)
)

# Validación Completa
print("\n Evaluando modelo YOLO en conjunto de TEST...")
metrics_yolo = model_yolo.val(split='test')

print(f"\nRESULTADOS YOLO:")
print(f" -> mAP50    (Precisión media @ IoU 0.5): {metrics_yolo.box.map50:.4f}")
print(f" -> mAP50-95 (Precisión media rigurosa):  {metrics_yolo.box.map:.4f}")

In [ ]:
results_path = 'runs/detect/train_implante/results.csv'

df = pd.read_csv(results_path)
df.columns = [x.strip() for x in df.columns]
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# Gráfica de Pérdidas (Loss)
axs[0].plot(df['train/box_loss'], label='Train Box Loss', color='blue')
axs[0].plot(df['train/cls_loss'], label='Train Cls Loss', color='orange')
axs[0].plot(df['val/box_loss'], label='Val Box Loss', color='cyan', linestyle='--')
axs[0].plot(df['val/cls_loss'], label='Val Cls Loss', color='red', linestyle='--')
axs[0].set_title('Curvas de Loss (YOLOv8)')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[0].grid(True)

# Gráfica de Métricas (mAP)
axs[1].plot(df['metrics/mAP50(B)'], label='mAP 50', color='green')
axs[1].plot(df['metrics/mAP50-95(B)'], label='mAP 50-95', color='purple')
axs[1].set_title('Métricas de Precisión (mAP)')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Score')
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()


## Detection learning curves

The plot shows training losses and validation mAP from the current run. Archived test mAP50 was 0.9950 and mAP50–95 was 0.7701 on 35 images. These values are specific to the archived split and heuristic annotations. The supplied data archive uses a different split; new runs must report their own metrics.


In [ ]:
# Visualización de predicciones
print("\n Visualización de Resultados en Test (Detección de Implantes) - Grid 3x3:")

test_images_path = os.path.join(BASE_DIR, 'test', 'images', '*.jpg')
test_images = glob.glob(test_images_path)

num_samples = 9
samples = random.sample(test_images, min(num_samples, len(test_images)))

plt.figure(figsize=(15, 15))

for i, img_path in enumerate(samples):
    results = model_yolo.predict(img_path, conf=0.25, verbose=False)

    res_plotted = results[0].plot()
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

    plt.subplot(3, 3, i+1)
    plt.imshow(res_rgb)
    plt.axis('off')
    plt.title(f"Test Sample {i+1}")

plt.tight_layout()
plt.show()


## Qualitative detection inspection

Predicted boxes support visual inspection of the current model. A few examples cannot establish generalization or clinical usefulness. Images are loaded only from the locally provided test split.


# Implant segmentation with U-Net

The second task uses a U-Net with an ImageNet-pretrained ResNet18 encoder, binary Dice loss and Adam. Images and masks share geometric augmentation. Training lasts 30 epochs at 256 × 256 resolution.

The cleaned edition uses Albumentations 2.x Gaussian-noise defaults. The archived notebook emitted a warning that its `var_limit` option was ignored; the exact installed version was not recorded. This is a documented compatibility adjustment, not an exact reconstruction of the original run.


In [ ]:

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from torch.utils.data import DataLoader, Dataset

# Definir las Transformaciones Para DataAumgentation
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(p=0.2),
    A.GaussNoise(p=0.2),  # Albumentations 2.x defaults; see docs/provenance.md
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], seed=42)

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Dataset
class MedicalSegmentationDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.img_dir = os.path.join(root_dir, split, 'images')
        self.mask_dir = os.path.join(root_dir, split, 'masks')
        self.transform = transform
        self.images = sorted(f for f in os.listdir(self.img_dir) if f.endswith('.jpg'))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_name = img_name.replace('.jpg', '.png')
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            mask = mask.float().unsqueeze(0) / 255.0
            mask = (mask > 0.5).float()

        return image, mask

# DataLoaders
train_ds = MedicalSegmentationDataset(BASE_DIR, 'train', transform=train_transform)
val_ds   = MedicalSegmentationDataset(BASE_DIR, 'val', transform=val_transform)
test_ds  = MedicalSegmentationDataset(BASE_DIR, 'test', transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False)

# Modelo, Loss y Optimizador
model_unet = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

criterion = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
optimizer = torch.optim.Adam(model_unet.parameters(), lr=0.0001)


# Bucle de Entrenamiento
def train_segmentation_epoch(model, loader, is_train=True):
    if is_train: model.train()
    else: model.eval()

    running_loss = 0.0
    running_acc = 0.0

    with torch.set_grad_enabled(is_train):
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)

            if is_train: optimizer.zero_grad()

            # Forward
            outputs = model(images)
            loss = criterion(outputs, masks)

            # Calculamos Accuracy del batch actual
            acc = calculate_accuracy(outputs, masks)

            # Backward
            if is_train:
                loss.backward()
                optimizer.step()

            running_loss += loss.item()
            running_acc += acc

    return running_loss / len(loader), running_acc / len(loader)

print("... Entrenando U-Net con Augmentation y Métricas ...")

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': []
}

epochs = 30

for epoch in range(epochs):
    # Entrenar
    t_loss, t_acc = train_segmentation_epoch(model_unet, train_loader, is_train=True)
    # Validar
    v_loss, v_acc = train_segmentation_epoch(model_unet, val_loader, is_train=False)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {t_loss:.4f} (T) {v_loss:.4f} (V) | Acc: {t_acc:.4f} (T) {v_acc:.4f} (V)")


In [ ]:
print("\n Análisis de Entrenamiento:")

plt.figure(figsize=(14, 5))

# Gráfica 1: Pérdida (Dice Loss)
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', color='blue')
plt.plot(history['val_loss'], label='Val Loss', color='orange')
plt.title("Evolución de Dice Loss (Menor es mejor)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# Gráfica 2: Accuracy (Precisión Píxel a Píxel)
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy', color='green')
plt.plot(history['val_acc'], label='Val Accuracy', color='red')
plt.title("Evolución de Accuracy (Mayor es mejor)")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (0-1)")
plt.legend()
plt.grid(True)

plt.show()

## Segmentation learning curves

The archived minimum validation loss was 0.0408 at epoch 29. The original code tested the final epoch-30 model, without selecting a best-validation checkpoint. This notebook preserves that evaluation choice. Pixel accuracy includes background and does not replace foreground Dice or IoU.


In [ ]:
print("\nEvaluación en el conjunto de TEST")
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)
test_loss, test_acc = train_segmentation_epoch(model_unet, test_loader, is_train=False)
print("\n" + "="*40)
print(f"RESULTADOS FINALES EN TEST")
print("="*40)
print(f"Test Dice Loss:   {test_loss:.4f}")
print(f"Test Pixel Acc:   {test_acc:.4f}  ({test_acc*100:.2f}%)")
print("="*40)

In [ ]:
print("Visualización de Resultados en Test:")
indices = random.sample(range(len(test_ds)), min(5, len(test_ds)))
for idx in indices:
    show_segmentation_result(model_unet, test_ds, idx)

## Interpreting segmentation outputs

Compare predictions with the automatically generated masks to inspect agreement and failure cases. The reported loss and accuracy are equal averages over batches, including the smaller last batch. The archived test loss was 0.0334 and pixel accuracy 0.9813. Do not infer a foreground Dice score by subtracting this loss from one. No patient-level split or expert annotation was established.
